In [1]:
# ============================================================================
# 03_judge_metric.ipynb  —  LLM judge + input-grounded faithfulness metric
# ----------------------------------------------------------------------------
# Single-cell notebook. All code and comments are in English.
# Figures: seaborn grayscale, no captions, dpi=600, saved as both PNG and PDF.
# Run order: 00 -> 01 -> 02 -> 03 -> 04 -> 05.
# ============================================================================
import sys, os, json, warnings
from pathlib import Path
import numpy as np
import pandas as pd

# Make src/ importable regardless of the notebook's working directory.
_HERE = Path.cwd()
_ROOT = _HERE if (_HERE / "src").exists() else _HERE.parent
sys.path.insert(0, str(_ROOT / "src"))
from common import (PROJECT_ROOT, DATA_DIR, RESULTS_DIR, FIGURES_DIR, TABLES_DIR,
                    load_env, set_grayscale_style, save_fig, save_table,
                    require_input, RANDOM_SEED)

import matplotlib.pyplot as plt
import seaborn as sns
set_grayscale_style()
np.random.seed(RANDOM_SEED)
warnings.filterwarnings("ignore")
# NOTE: absolute local paths are never printed (anonymity for double-blind review).

# ----------------------------------------------------------------------------
# PURPOSE
#   Score every perturbed output on two independent channels:
#     JUDGE   : LLM judge, 5 integer axes (1-5). The qualitative-adequacy axis is
#               the one technical metrics cannot measure; it is the central axis.
#     METRIC  : faithfulness grounded in the FIXED quantitative input (fraction of
#               the output's factual claims entailed by the financial profile).
#   Judge is run INFORMED (told the gap) and BLIND (not told) for the ablation.
#   A secondary judge from a different family re-scores a random sample for
#   inter-judge agreement (Spearman rho, quadratic-weighted kappa).
#
# INPUT   data/perturbed.parquet
# OUTPUT  data/scored.parquet  (adds judge_qual, judge_axes, faithfulness, judge_cond)
# ----------------------------------------------------------------------------
cfg = load_env()
from openai import OpenAI
client = OpenAI(api_key=cfg["api_key"])
JUDGE_PRIMARY = cfg["strong"]
JUDGE_SECONDARY = cfg["mid"]       # different tier/family for agreement check

pert = pd.read_parquet(require_input(DATA_DIR / "perturbed.parquet",
                                     hint="Run 02_perturbation first."))

def judge(rationale, rating, delta, informed, model):
    gap_ctx = (f"Context: the agency rating is {int(delta):+d} notches vs the "
               f"quantitative profile; a good rationale presents a qualitative "
               f"justification whose direction matches this gap."
               if informed else
               "Score the rationale on its own internal validity.")
    sys_msg = ("You are evaluating a credit rating rationale. Score five axes "
               "1-5: grade_accuracy, grade_rationale_consistency, "
               "qualitative_adequacy, factual_faithfulness, explainability. "
               "qualitative_adequacy: 5=clear & consistent with the gap's "
               "direction, 3=present but generic, 1=absent or contradicting.")
    user = (f"{gap_ctx}\nRating: {rating}\nRationale:\n{rationale}\n\n"
            f"Return JSON with the five integer axes.")
    r = client.chat.completions.create(model=model, temperature=0.0,
        messages=[{"role":"system","content":sys_msg},
                  {"role":"user","content":user}],
        response_format={"type":"json_object"})
    return json.loads(r.choices[0].message.content)

def faithfulness(rationale, row):
    # Input-grounded: ask the model what fraction of the output's quantitative
    # claims are entailed by the fixed financial profile. (Proxy implementation.)
    sys_msg = ("Return JSON {\"entailed\": int, \"total\": int}: of the "
               "quantitative/factual claims in the rationale, how many are "
               "entailed by the stated financial sector/profile.")
    r = client.chat.completions.create(model=JUDGE_SECONDARY, temperature=0.0,
        messages=[{"role":"system","content":sys_msg},
                  {"role":"user","content":str(rationale)[:4000]}],
        response_format={"type":"json_object"})
    o = json.loads(r.choices[0].message.content)
    tot = max(1, int(o.get("total", 1)))
    return int(o.get("entailed", 0)) / tot

rows = []
for _, r in pert.iterrows():
    for informed in (True, False):
        try:
            ax = judge(r.rationale, r.rating, r.delta, informed, JUDGE_PRIMARY)
        except Exception as e:
            ax = {"qualitative_adequacy": np.nan}
        rows.append({"case_id": r.case_id, "condition": r.condition,
                     "pert": r.pert, "k": r.k,
                     "judge_cond": "informed" if informed else "blind",
                     "judge_qual": ax.get("qualitative_adequacy", np.nan),
                     "judge_axes": json.dumps(ax),
                     "faithfulness": faithfulness(r.rationale, r)})
scored = pd.DataFrame(rows)
scored.to_parquet(DATA_DIR / "scored.parquet", index=False)
print(scored.groupby(["pert","judge_cond"])["judge_qual"].mean().round(2))


pert  judge_cond
P0    blind         4.55
      informed      4.10
P1    blind         4.33
      informed      3.92
P2    blind         4.25
      informed      3.65
P3    blind         4.32
      informed      4.08
P4    blind         2.92
      informed      2.10
P5    blind         4.08
      informed      4.03
Name: judge_qual, dtype: float64
